# 03 – Alimentación a Parquet

Este notebook procesa el archivo:

`../data/raw/sobrantes y dietas 2025.xlsx`

y construye datasets estructurados para integrarlos más adelante al pipeline principal.

## Objetivos

- leer las hojas mensuales de alimentación
- normalizar columnas y fechas
- construir un dataset diario por corral
- leer correctamente las hojas de dieta por periodo
- documentar la vigencia correcta de cada dieta
- exportar resultados a parquet

## Aclaración importante sobre las hojas de dieta

La interpretación correcta del archivo es:

- **`dieta Mayo`** corresponde al periodo **enero a mayo**
- **`Dieta Junio`** corresponde al periodo **junio a julio**
- **`Dieta Agosto`** corresponde al periodo **agosto a septiembre**

Es decir, los nombres de esas hojas no indican un solo mes, sino el **inicio o referencia del periodo de dieta vigente**.


## 1. Importaciones


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)


## 2. Rutas del proyecto


In [5]:
INPUT_PATH = "../data/raw/dieta/sobrantes y dietas 2025.xlsx"

OUT_FEEDING = "../data/interim/feeding_daily_corral.parquet"
OUT_DIETS = "../data/interim/diet_periods.parquet"
OUT_DIET_SUMMARY = "../data/interim/diet_period_summary.parquet"

print("Archivo de entrada:", INPUT_PATH)


Archivo de entrada: ../data/raw/dieta/sobrantes y dietas 2025.xlsx


## 3. Inspección de hojas disponibles


In [6]:
xls = pd.ExcelFile(INPUT_PATH)

print("Hojas disponibles:")
display(pd.DataFrame({"sheet_name": xls.sheet_names}))


Hojas disponibles:


,sheet_name
0,Enero
1,Febrero
2,Marzo
3,Abril
4,Mayo
5,Junio
6,Julio
7,Agosto
8,Septiembre
9,dieta Mayo


## 4. Definición de hojas mensuales y hojas de dieta

Las hojas mensuales contienen observaciones diarias.  
Las hojas de dieta contienen la composición vigente por periodo.


In [7]:
monthly_sheets = [
    "Enero ", "Febrero", "Marzo", "Abril", "Mayo",
    "Junio", "Julio", "Agosto", "Septiembre"
]

diet_periods_map = {
    "dieta Mayo":   ("2025-01-01", "2025-05-31"),
    "Dieta Junio":  ("2025-06-01", "2025-07-31"),
    "Dieta Agosto": ("2025-08-01", "2025-09-30"),
}

print("Hojas mensuales:")
print(monthly_sheets)

print("\nHojas de dieta y vigencia:")
for k, v in diet_periods_map.items():
    print(k, "->", v)


Hojas mensuales:
['Enero ', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre']

Hojas de dieta y vigencia:
dieta Mayo -> ('2025-01-01', '2025-05-31')
Dieta Junio -> ('2025-06-01', '2025-07-31')
Dieta Agosto -> ('2025-08-01', '2025-09-30')


## 5. Funciones auxiliares

Se definen funciones para:

- limpiar nombres de columnas
- detectar encabezados en hojas mensuales
- detectar el encabezado real en hojas de dieta
- homogeneizar nombres de columnas


In [8]:
def clean_columns(cols):
    s = pd.Index(cols).astype(str)
    s = (
        s.str.strip()
         .str.lower()
         .str.replace("\n", " ", regex=False)
         .str.replace("  ", " ", regex=False)
         .str.replace(" ", "_", regex=False)
         .str.replace("°", "n", regex=False)
         .str.replace("º", "n", regex=False)
         .str.replace("%", "pct", regex=False)
         .str.replace("/", "_", regex=False)
         .str.replace(".", "", regex=False)
         .str.replace("-", "_", regex=False)
         .str.replace("__", "_", regex=False)
    )
    return list(s)

def try_read_sheet(file_path, sheet_name, header_candidates=(0, 1, 2, 3, 4, 5)):
    best_df = None
    best_score = -1

    expected_keywords = ["fecha", "corral", "corrales", "kg", "sobrante", "consumo"]

    for h in header_candidates:
        try:
            df = pd.read_excel(file_path, sheet_name=sheet_name, header=h)
            df.columns = clean_columns(df.columns)
            joined = " | ".join(df.columns)
            score = sum(kw in joined for kw in expected_keywords)
            if score > best_score:
                best_score = score
                best_df = df.copy()
        except Exception:
            pass

    return best_df, best_score

def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def read_diet_sheet(file_path, sheet_name):
    raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

    header_idx = None
    for i, row in raw.iterrows():
        row_str = row.astype(str).str.strip().str.upper().tolist()
        if "INGREDIENTES" in row_str:
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(f"No se encontró la fila de encabezado real en la hoja: {sheet_name}")

    header = raw.iloc[header_idx].astype(str).str.strip().tolist()
    df = raw.iloc[header_idx + 1:].copy()
    df.columns = clean_columns(header)

    return df, header_idx


## 6. Lectura de hojas mensuales de alimentación

Aquí se construye un dataframe diario por corral a partir de las hojas:

- Enero
- Febrero
- Marzo
- Abril
- Mayo
- Junio
- Julio
- Agosto
- Septiembre


In [9]:
feeding_frames = []
sheet_debug = []

for sheet in monthly_sheets:
    df, score = try_read_sheet(INPUT_PATH, sheet)

    if df is None:
        print(f"No se pudo leer la hoja: {sheet}")
        continue

    raw_cols = list(df.columns)

    fecha_col = first_existing(df, ["fecha"])
    corral_col = first_existing(df, ["corrales", "corral", "corral_id"])

    rename_map = {}

    if fecha_col:
        rename_map[fecha_col] = "date"
    if corral_col:
        rename_map[corral_col] = "corral_id"

    for old, new in [
        ("kg_am", "kg_am"),
        ("kg_pm", "kg_pm"),
        ("kg_totales", "kg_totales"),
        ("kg_total", "kg_totales"),
        ("sobrante", "sobrante"),
        ("consumo", "consumo"),
        ("rechazo", "rechazo"),
        ("n_de_vacas", "n_vacas"),
        ("no_de_vacas", "n_vacas"),
        ("n_vacas", "n_vacas"),
        ("kg_consumido_por_vaca", "kg_consumido_por_vaca"),
        ("promedio", "promedio_corral"),
        ("promedio_corral", "promedio_corral"),
    ]:
        if old in df.columns:
            rename_map[old] = new

    df = df.rename(columns=rename_map)

    keep = [
        c for c in [
            "date", "corral_id", "kg_am", "kg_pm", "kg_totales",
            "sobrante", "consumo", "rechazo", "n_vacas",
            "kg_consumido_por_vaca", "promedio_corral"
        ] if c in df.columns
    ]

    df = df[keep].copy()

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce").ffill()
        df["date"] = pd.to_datetime(df["date"]).dt.normalize()

    if "corral_id" in df.columns:
        df["corral_id"] = df["corral_id"].astype(str).str.strip()
        df = df[df["corral_id"].notna()]
        df = df[~df["corral_id"].str.lower().isin(["nan", "none", "total", "totales"])]

    for c in ["kg_am", "kg_pm", "kg_totales", "sobrante", "consumo", "rechazo", "n_vacas", "kg_consumido_por_vaca", "promedio_corral"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["source_sheet"] = sheet.strip()
    feeding_frames.append(df)

    sheet_debug.append({
        "sheet": sheet,
        "score": score,
        "rows": len(df),
        "columns": ", ".join(raw_cols[:20])
    })

sheet_debug_df = pd.DataFrame(sheet_debug)
display(sheet_debug_df)

feeding_daily_corral = pd.concat(feeding_frames, ignore_index=True)

print("Shape feeding_daily_corral:", feeding_daily_corral.shape)
display(feeding_daily_corral.head(20))


,sheet,score,rows,columns
0,Enero,6,186,"fecha, corrales, kg_am, kg_pm, kg_totales, sob..."
1,Febrero,6,168,"fecha, corrales, kg_am, kg_pm, kg_totales, sob..."
2,Marzo,6,186,"fecha, corrales, kg_am, kg_pm, kg_totales, sob..."
3,Abril,6,180,"fecha, corrales, kg_am, kg_pm, kg_totales, sob..."
4,Mayo,6,186,"fecha, corrales, kg_am, kg_pm, kg_totales, sob..."
5,Junio,6,180,"fecha, corrales, kg_am, kg_pm, kg_totales, sob..."
6,Julio,6,186,"fecha, corrales, kg_am, kg_pm, kg_totales, sob..."
7,Agosto,6,186,"fecha, corrales, kg_am, kg_pm, kg_totales, sob..."
8,Septiembre,6,180,"fecha, corrales, kg_am, kg_pm, kg_totales, sob..."


Shape feeding_daily_corral: (1638, 11)


,date,corral_id,kg_am,kg_pm,kg_totales,sobrante,consumo,source_sheet,rechazo,kg_consumido_por_vaca,promedio_corral
0,2025-01-01,1.0,500.0,500.0,1000.0,0.0,1000.0,Enero,NaN,NaN,NaN
1,2025-01-01,2.0,650.0,650.0,1300.0,0.0,1300.0,Enero,NaN,NaN,NaN
2,2025-01-01,3.0,600.0,600.0,1200.0,0.0,1200.0,Enero,NaN,NaN,NaN
3,2025-01-01,4.0,650.0,650.0,1300.0,10.0,1290.0,Enero,NaN,NaN,NaN
4,2025-01-01,5.0,600.0,600.0,1200.0,0.0,1200.0,Enero,NaN,NaN,NaN
5,2025-01-01,6.0,1000.0,1000.0,2000.0,0.0,2000.0,Enero,NaN,NaN,NaN
6,2025-01-02,1.0,550.0,550.0,1100.0,110.0,990.0,Enero,NaN,NaN,NaN
7,2025-01-02,2.0,650.0,650.0,1300.0,130.0,1170.0,Enero,NaN,NaN,NaN
8,2025-01-02,3.0,600.0,600.0,1200.0,80.0,1120.0,Enero,NaN,NaN,NaN
9,2025-01-02,4.0,650.0,650.0,1300.0,220.0,1080.0,Enero,NaN,NaN,NaN


### Comentarios y observaciones

En esta etapa conviene revisar:

- si todas las hojas se leyeron
- si `date` y `corral_id` quedaron bien pobladas
- si los nombres de columnas quedaron homogéneos
- si existen cambios de formato entre meses

Las hojas mensuales pueden tener encabezados ligeramente distintos, por eso este parser intenta ser tolerante.


## 7. Limpieza adicional y variables derivadas de alimentación


In [10]:
feeding_daily_corral = feeding_daily_corral.dropna(subset=["date", "corral_id"]).copy()

if "kg_totales" in feeding_daily_corral.columns and "n_vacas" in feeding_daily_corral.columns:
    feeding_daily_corral["kg_ofrecidos_por_vaca"] = (
        feeding_daily_corral["kg_totales"] / feeding_daily_corral["n_vacas"].replace(0, np.nan)
    )

if "sobrante" in feeding_daily_corral.columns and "kg_totales" in feeding_daily_corral.columns:
    feeding_daily_corral["sobrante_pct"] = (
        feeding_daily_corral["sobrante"] / feeding_daily_corral["kg_totales"].replace(0, np.nan)
    )

if "consumo" in feeding_daily_corral.columns and "kg_totales" in feeding_daily_corral.columns:
    feeding_daily_corral["consumo_pct"] = (
        feeding_daily_corral["consumo"] / feeding_daily_corral["kg_totales"].replace(0, np.nan)
    )

print("Shape después de limpieza:", feeding_daily_corral.shape)
display(feeding_daily_corral.head(20))


Shape después de limpieza: (1638, 13)


,date,corral_id,kg_am,kg_pm,kg_totales,sobrante,consumo,source_sheet,rechazo,kg_consumido_por_vaca,promedio_corral,sobrante_pct,consumo_pct
0,2025-01-01,1.0,500.0,500.0,1000.0,0.0,1000.0,Enero,NaN,NaN,NaN,0.000000,1.000000
1,2025-01-01,2.0,650.0,650.0,1300.0,0.0,1300.0,Enero,NaN,NaN,NaN,0.000000,1.000000
2,2025-01-01,3.0,600.0,600.0,1200.0,0.0,1200.0,Enero,NaN,NaN,NaN,0.000000,1.000000
3,2025-01-01,4.0,650.0,650.0,1300.0,10.0,1290.0,Enero,NaN,NaN,NaN,0.007692,0.992308
4,2025-01-01,5.0,600.0,600.0,1200.0,0.0,1200.0,Enero,NaN,NaN,NaN,0.000000,1.000000
5,2025-01-01,6.0,1000.0,1000.0,2000.0,0.0,2000.0,Enero,NaN,NaN,NaN,0.000000,1.000000
6,2025-01-02,1.0,550.0,550.0,1100.0,110.0,990.0,Enero,NaN,NaN,NaN,0.100000,0.900000
7,2025-01-02,2.0,650.0,650.0,1300.0,130.0,1170.0,Enero,NaN,NaN,NaN,0.100000,0.900000
8,2025-01-02,3.0,600.0,600.0,1200.0,80.0,1120.0,Enero,NaN,NaN,NaN,0.066667,0.933333
9,2025-01-02,4.0,650.0,650.0,1300.0,220.0,1080.0,Enero,NaN,NaN,NaN,0.169231,0.830769


## 8. Revisión rápida del dataset diario por corral


In [11]:
print("Rango de fechas:", feeding_daily_corral["date"].min(), "->", feeding_daily_corral["date"].max())
print("Número de corrales:", feeding_daily_corral["corral_id"].nunique())
print("Fechas únicas:", feeding_daily_corral["date"].nunique())

display(
    feeding_daily_corral.groupby(feeding_daily_corral["date"].dt.to_period("M"))["date"].nunique()
)


Rango de fechas: 0025-07-02 00:00:00 -> 2025-09-30 00:00:00
Número de corrales: 12
Fechas únicas: 271


date
25-07       5
2025-01    31
2025-02    25
2025-03    30
2025-04    29
2025-05    30
2025-06    29
2025-07    31
2025-08    31
2025-09    30
Freq: M, Name: date, dtype: int64

### Comentarios y observaciones

Este dataset ya queda a nivel:

```text
date + corral_id
```

Todavía no está a nivel vaca.  
Para llevarlo al entrenamiento por vaca será necesario un mapeo:

```text
cow_id + date -> corral_id
```

Si no se tiene ese mapeo, solo podrá unirse a nivel global por fecha o por grupo de vacas.


## 9. Lectura de hojas de dieta por periodo

Estas hojas no representan observaciones diarias, sino la dieta vigente en bloques temporales.

### Interpretación correcta

- `dieta Mayo` -> vigente de enero a mayo
- `Dieta Junio` -> vigente de junio a julio
- `Dieta Agosto` -> vigente de agosto a septiembre

A diferencia de las hojas mensuales, aquí se detecta de forma explícita la fila que contiene `INGREDIENTES`, porque ese es el encabezado real de la tabla.


In [12]:
diet_frames = []
diet_debug = []

for sheet, (start_date, end_date) in diet_periods_map.items():
    df, header_idx = read_diet_sheet(INPUT_PATH, sheet)

    print(f"\nHoja: {sheet}")
    print("Fila de encabezado detectada:", header_idx)
    print("Columnas detectadas:")
    print(df.columns.tolist())
    display(df.head(15))

    rename_map = {}

    ing_col = first_existing(df, ["ingredientes", "ingrediente"])
    pct_col = first_existing(df, ["pct_de_ms", "pct_ms", "_de_ms", "de_ms"])
    hum_col = first_existing(df, ["humeda", "kg_humeda"])
    sec_col = first_existing(df, ["seca", "kg_seca"])
    ampm_col = first_existing(df, ["am_pm", "am__pm"])

    if ing_col: rename_map[ing_col] = "ingrediente"
    if pct_col: rename_map[pct_col] = "pct_ms"
    if hum_col: rename_map[hum_col] = "kg_humeda"
    if sec_col: rename_map[sec_col] = "kg_seca"
    if ampm_col: rename_map[ampm_col] = "am_pm"

    df = df.rename(columns=rename_map)

    keep = [c for c in ["ingrediente", "pct_ms", "kg_humeda", "kg_seca", "am_pm"] if c in df.columns]
    df = df[keep].copy()

    if "ingrediente" in df.columns:
        df["ingrediente"] = df["ingrediente"].astype(str).str.strip()
        df = df[df["ingrediente"].notna()]
        df = df[~df["ingrediente"].str.lower().isin(["nan", "none", ""])]
        df = df[~df["ingrediente"].str.contains("total", case=False, na=False)]
        df = df[~df["ingrediente"].str.contains("ingredientes", case=False, na=False)]

    for c in ["pct_ms", "kg_humeda", "kg_seca", "am_pm"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["diet_period"] = sheet
    df["period_start"] = pd.to_datetime(start_date)
    df["period_end"] = pd.to_datetime(end_date)

    print("Después de limpiar:")
    display(df.head(20))
    print("Rows:", len(df))

    diet_frames.append(df)

    diet_debug.append({
        "sheet": sheet,
        "header_idx": header_idx,
        "rows": len(df),
        "period_start": start_date,
        "period_end": end_date,
        "columns_finales": ", ".join(df.columns)
    })

diet_debug_df = pd.DataFrame(diet_debug)
display(diet_debug_df)

diet_periods = pd.concat(diet_frames, ignore_index=True)

print("Shape diet_periods:", diet_periods.shape)
display(diet_periods.head(30))



Hoja: dieta Mayo
Fila de encabezado detectada: 4
Columnas detectadas:
['ingredientes', nan, nan, nan, '05']


,ingredientes,NaN,NaN,NaN,05
5,TRITICALE,0.31,25.16,7.7996,1937.32
6,NaN,NaN,NaN,0,0
7,PATA DE CEBADA,0.88,0.46,0.4048,35.42
8,NaN,NaN,NaN,0,0
9,ORO MILK,0.9,15.76,14.184,1213.52
10,Melaza(50% Agua),0.4,3.33,1.332,256.41
11,NaN,NaN,NaN,NaN,NaN
12,Agua,0.001,5.5,0.0055,423.5
13,Total Dieta Integral,NaN,50.21,23.7259,3866.17
14,NaN,NaN,NaN,NaN,NaN


Después de limpiar:


,ingrediente,diet_period,period_start,period_end
5,TRITICALE,dieta Mayo,2025-01-01,2025-05-31
7,PATA DE CEBADA,dieta Mayo,2025-01-01,2025-05-31
9,ORO MILK,dieta Mayo,2025-01-01,2025-05-31
10,Melaza(50% Agua),dieta Mayo,2025-01-01,2025-05-31
12,Agua,dieta Mayo,2025-01-01,2025-05-31
15,TRITICALE,dieta Mayo,2025-01-01,2025-05-31


Rows: 6

Hoja: Dieta Junio
Fila de encabezado detectada: 3
Columnas detectadas:
['ingredientes', nan, nan, nan, '05']


...

,ingrediente,diet_period,period_start,period_end
4,SILO DE AVENA,Dieta Agosto,2025-08-01,2025-09-30
5,silo de MAIZ,Dieta Agosto,2025-08-01,2025-09-30
6,PATA DE CEBADA,Dieta Agosto,2025-08-01,2025-09-30
7,MAIZ MOLIDO,Dieta Agosto,2025-08-01,2025-09-30
8,Pasta se SOYA,Dieta Agosto,2025-08-01,2025-09-30
9,ORO Balance,Dieta Agosto,2025-08-01,2025-09-30
10,ORO MILK,Dieta Agosto,2025-08-01,2025-09-30
11,Melaza(50% Agua),Dieta Agosto,2025-08-01,2025-09-30
13,Agua,Dieta Agosto,2025-08-01,2025-09-30


Rows: 9


,sheet,header_idx,rows,period_start,period_end,columns_finales
0,dieta Mayo,4,6,2025-01-01,2025-05-31,"ingrediente, diet_period, period_start, period..."
1,Dieta Junio,3,6,2025-06-01,2025-07-31,"ingrediente, diet_period, period_start, period..."
2,Dieta Agosto,3,9,2025-08-01,2025-09-30,"ingrediente, diet_period, period_start, period..."


Shape diet_periods: (21, 4)


,ingrediente,diet_period,period_start,period_end
0,TRITICALE,dieta Mayo,2025-01-01,2025-05-31
1,PATA DE CEBADA,dieta Mayo,2025-01-01,2025-05-31
2,ORO MILK,dieta Mayo,2025-01-01,2025-05-31
3,Melaza(50% Agua),dieta Mayo,2025-01-01,2025-05-31
4,Agua,dieta Mayo,2025-01-01,2025-05-31
5,TRITICALE,dieta Mayo,2025-01-01,2025-05-31
6,TRITICALE,Dieta Junio,2025-06-01,2025-07-31
7,PATA DE CEBADA,Dieta Junio,2025-06-01,2025-07-31
8,ORO Balance,Dieta Junio,2025-06-01,2025-07-31
9,ORO MILK,Dieta Junio,2025-06-01,2025-07-31


## 10. Resumen de dieta por periodo


In [13]:
summary_rows = []

for period_name, g in diet_periods.groupby("diet_period"):
    row = {
        "diet_period": period_name,
        "period_start": g["period_start"].iloc[0],
        "period_end": g["period_end"].iloc[0],
        "n_ingredientes": g["ingrediente"].nunique() if "ingrediente" in g.columns else np.nan,
        "diet_dm_total": g["kg_seca"].sum(skipna=True) if "kg_seca" in g.columns else np.nan,
        "diet_wet_total": g["kg_humeda"].sum(skipna=True) if "kg_humeda" in g.columns else np.nan,
        "diet_pct_ms_total": g["pct_ms"].sum(skipna=True) if "pct_ms" in g.columns else np.nan,
    }

    ingredient_list = g["ingrediente"].dropna().astype(str).str.lower().tolist() if "ingrediente" in g.columns else []

    for key, token in {
        "usa_oro_milk": "oro milk",
        "usa_oro_balance": "oro balance",
        "usa_silo_maiz": "silo ma",
        "usa_silo_avena": "avena",
        "usa_ensilado": "ensil",
        "usa_heno": "heno",
        "usa_triticale": "triticale",
        "usa_melaza": "melaza",
    }.items():
        row[key] = int(any(token in ing for ing in ingredient_list))

    summary_rows.append(row)

diet_period_summary = pd.DataFrame(summary_rows).sort_values("period_start").reset_index(drop=True)

display(diet_period_summary)


,diet_period,period_start,period_end,n_ingredientes,diet_dm_total,diet_wet_total,diet_pct_ms_total,usa_oro_milk,usa_oro_balance,usa_silo_maiz,usa_silo_avena,usa_ensilado,usa_heno,usa_triticale,usa_melaza
0,dieta Mayo,2025-01-01,2025-05-31,5,NaN,NaN,NaN,1,0,0,0,0,0,1,1
1,Dieta Junio,2025-06-01,2025-07-31,6,NaN,NaN,NaN,1,1,0,0,0,0,1,1
2,Dieta Agosto,2025-08-01,2025-09-30,9,NaN,NaN,NaN,1,1,0,1,0,0,0,1


### Comentarios y observaciones

Este resumen no reemplaza la tabla detallada de ingredientes, pero sí facilita mucho el uso de la dieta como variable de periodo dentro del modelo.


## 11. Guardado a parquet


In [14]:
Path(OUT_FEEDING).parent.mkdir(parents=True, exist_ok=True)

feeding_daily_corral.to_parquet(OUT_FEEDING, index=False)
diet_periods.to_parquet(OUT_DIETS, index=False)
diet_period_summary.to_parquet(OUT_DIET_SUMMARY, index=False)

print("Guardado:")
print(OUT_FEEDING)
print(OUT_DIETS)
print(OUT_DIET_SUMMARY)


Guardado:
../data/interim/feeding_daily_corral.parquet
../data/interim/diet_periods.parquet
../data/interim/diet_period_summary.parquet


## 12. Conclusiones

En este notebook se construyeron tres datasets útiles:

### 1. `feeding_daily_corral`
Observaciones diarias por fecha y corral, útiles para modelar oferta, consumo, sobrante y rechazo.

### 2. `diet_periods`
Tabla detallada de ingredientes por periodo de dieta vigente.

### 3. `diet_period_summary`
Resumen compacto por periodo, útil para merges posteriores y creación de features.

## Observación importante

Para usar estos datos dentro del dataset de entrenamiento por vaca se necesitará:

- un mapeo vaca -> corral por fecha, o
- una agregación global por fecha si no existe ese mapeo.

## Mejora aplicada en esta versión

Las hojas de dieta ahora se leen buscando explícitamente la fila donde aparece `INGREDIENTES`, evitando que el encabezado se interprete mal y dejando de generar tablas vacías.

## Próximos pasos sugeridos

1. validar columnas reales de cada hoja mensual  
2. revisar si todos los corrales quedaron bien parseados  
3. integrar `feeding_daily_corral` al pipeline principal  
4. construir features de alimentación para el entrenamiento
